In [ ]:
# Resolve repository imports and data paths when opened from notebooks/.
from pathlib import Path
import os
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "sae").is_dir():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Run this notebook from the KronSAE repository.")
    repo_root = repo_root.parent

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
!pip install scikit-learn
!pip install seaborn
%pip install -r requirements.txt

In [ ]:
from tqdm.notebook import tqdm

import torch
import numpy as np
    
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils
from transformers import AutoConfig
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from safetensors.torch import load_model
##########
import torch
import json
import math
import einops 
import numpy as np
import torch.nn.functional as F

from torch import nn
from transformer_lens import HookedTransformer
from transformer_lens import utils

from sae import KronSAE, TopKSAE
from sae.config import TrainingConfig, SAEConfig

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import HookedTransformer, FactoredMatrix

import einops
from fancy_einsum import einsum
import tqdm.auto as tqdm
import plotly.express as px

from functools import partial

from huggingface_hub.hf_api import HfFolder

device = "cuda"
hook_point = "resid_post"
layer_num = 5
torch.set_grad_enabled(False)

In [ ]:
model = HookedTransformer.from_pretrained("Qwen/Qwen2.5-1.5B", dtype=torch.bfloat16).to(device)

In [ ]:
def get_sae(sae_type: str, path):
    # Load transformer model
    sae_class = {
        "topk": TopKSAE,
        "kronsae": KronSAE
    }[sae_type]
    
    sae_config = SAEConfig.from_pretrained(path)
    sae_config.dtype = "float32"
    sae_config.sae_dtype = "float32"

    sae = sae_class(sae_config)
    load_model(sae, f"{path}/model.safetensors")
    sae.to(device)
    return {'sae': sae, 'config': sae_config}
    
# sae_paths = {
#     "topk" : "saes/16k_topk", 
#     "kronsae" : "saes/16k_kronsae_4_16",
# }
d = get_sae("topk", "saes/1_5b_qwen/32k_topk")
topk_sae = d["sae"]
topk_sae_config = d["config"]


In [ ]:
dataset = load_dataset("HuggingFaceFW/fineweb-edu", split="train", name="CC-MAIN-2024-51", streaming=True, trust_remote_code=True)

In [ ]:
import torch
from datasets import load_dataset

# --- CONFIG ----
device = torch.device(device) #"cuda" if torch.cuda.is_available() else "cpu")
hook_point = utils.get_act_name("resid_post", 14)
# topk_sae = ...        # your SAE encoder, already on `device`
# model = ...           # your model, already on `device`

# --- RUNNING STATISTICS STATE ---
mu = None           # Tensor[D], on `device`
M2 = None           # Tensor[D, D], on `device`
n  = 0              # total samples seen

@torch.no_grad()  # disable gradients inside hook
def update_online_cov(x: torch.Tensor):
    """
    x: Tensor of shape (N_batch, D) already on `device` and float32.
    Updates global mu, M2, and n in-place.
    """
    global mu, M2, n

    N, D = x.shape
    n_new = n + N

    if mu is None:
        # First batch
        mu = x.mean(dim=0)                               # [D]
        M2 = (x - mu).t().mm(x - mu)                     # [D, D]
        n = N
        return

    # batch mean
    mb = x.mean(dim=0)                                   # [D]
    delta = mb - mu                                      # [D]
    mu = mu + delta * (N / n_new)                        # [D]

    # within-batch scatter
    dev = x - mb                                         # [N, D]
    M2_batch = dev.t().mm(dev)                           # [D, D]

    # cross term
    cross = torch.outer(delta, delta) * (n * N / n_new)  # [D, D]

    # in-place update
    M2 += M2_batch
    M2 += cross

    n = n_new

def resid_post_hook(resid_post, hook):
    # resid_post: [B, T, D], float32 on CPU by default?
    B, T, D = resid_post.shape
    # encode directly on `device`
    flat = resid_post.view(B * T, D).float().to(device)
    hidden = topk_sae.encode(flat)                       # already on device
    update_online_cov(hidden)                            # fast, no transfers
    return resid_post

# --- MAIN LOOP ---
model.to(device)
topk_sae.to(device)
model.eval()                                            # switch to eval mode

# replace streaming=True if you need it
# dataset = load_dataset("HuggingFaceFW/fineweb-edu",
#                        split="train",
#                        streaming=True,
#                        trust_remote_code=True)
iter_ds = iter(dataset)

num_iters = 5000
with torch.no_grad():  # full-loop no-grad
    for _ in tqdm.tqdm(range(num_iters)):
        sample = next(iter_ds)
        tokens = model.to_tokens(sample["text"]).to(device)
        _ = model.run_with_hooks(
            tokens,
            fwd_hooks=[(hook_point, resid_post_hook)]
        )

# --- FINALIZE ---
cov = M2 / (n - 1)                                      # unbiased covariance
std = torch.sqrt(torch.diag(cov))                      # [D]
corrA = cov / (std[:, None] * std[None, :] + 1e-6)      # [D, D]


In [ ]:
d = get_sae("topk", "saes/1_5b_qwen/32k_topk")
topk_sae_random = d["sae"]
torch.nn.init.uniform_(
                topk_sae_random.W_dec
            )
torch.nn.init.uniform_(
                topk_sae_random.W_enc
            )
torch.nn.init.uniform_(
                topk_sae_random.b_dec
            )
torch.nn.init.uniform_(
                topk_sae_random.b_enc
            )

In [ ]:
import torch
from datasets import load_dataset

# --- CONFIG ----
device = torch.device(device) #"cuda" if torch.cuda.is_available() else "cpu")
hook_point = utils.get_act_name("resid_post", 14)
# topk_sae = ...        # your SAE encoder, already on `device`
# model = ...           # your model, already on `device`

# --- RUNNING STATISTICS STATE ---
mu = None           # Tensor[D], on `device`
M2 = None           # Tensor[D, D], on `device`
n  = 0              # total samples seen

@torch.no_grad()  # disable gradients inside hook
def update_online_cov(x: torch.Tensor):
    """
    x: Tensor of shape (N_batch, D) already on `device` and float32.
    Updates global mu, M2, and n in-place.
    """
    global mu, M2, n

    N, D = x.shape
    n_new = n + N

    if mu is None:
        # First batch
        mu = x.mean(dim=0)                               # [D]
        M2 = (x - mu).t().mm(x - mu)                     # [D, D]
        n = N
        return

    # batch mean
    mb = x.mean(dim=0)                                   # [D]
    delta = mb - mu                                      # [D]
    mu = mu + delta * (N / n_new)                        # [D]

    # within-batch scatter
    dev = x - mb                                         # [N, D]
    M2_batch = dev.t().mm(dev)                           # [D, D]

    # cross term
    cross = torch.outer(delta, delta) * (n * N / n_new)  # [D, D]

    # in-place update
    M2 += M2_batch
    M2 += cross

    n = n_new

def resid_post_hook(resid_post, hook):
    # resid_post: [B, T, D], float32 on CPU by default?
    B, T, D = resid_post.shape
    # encode directly on `device`
    flat = resid_post.view(B * T, D).float().to(device)
    hidden = topk_sae_random.encode(flat)                       # already on device
    update_online_cov(hidden)                            # fast, no transfers
    return resid_post

# --- MAIN LOOP ---
model.to(device)
topk_sae.to(device)
model.eval()                                            # switch to eval mode

# replace streaming=True if you need it
# dataset = load_dataset("HuggingFaceFW/fineweb-edu",
#                        split="train",
#                        streaming=True,
#                        trust_remote_code=True)
iter_ds = iter(dataset)

num_iters = 5000
with torch.no_grad():  # full-loop no-grad
    for _ in tqdm.tqdm(range(num_iters)):
        sample = next(iter_ds)
        tokens = model.to_tokens(sample["text"]).to(device)
        _ = model.run_with_hooks(
            tokens,
            fwd_hooks=[(hook_point, resid_post_hook)]
        )

# --- FINALIZE ---
cov = M2 / (n - 1)                                      # unbiased covariance
std = torch.sqrt(torch.diag(cov))                      # [D]
corrA_rand = cov / (std[:, None] * std[None, :] + 1e-6)      # [D, D]


In [ ]:
#torch.save(corr, "topk_corr.pt")

In [ ]:
sae_paths = {
    "topk" : "saes/16k_topk", 
    "kronsae" : "saes/16k_kronsae_4_16",
}
d = get_sae("kronsae", "saes/1_5b_qwen/32k_kronsae")
kronsae = d["sae"]
kronsae_config = d["config"]

In [ ]:
import torch
from datasets import load_dataset

# --- CONFIG ----
device = torch.device(device) #"cuda" if torch.cuda.is_available() else "cpu")
hook_point = utils.get_act_name("resid_post", 14)
# topk_sae = ...        # your SAE encoder, already on `device`
# model = ...           # your model, already on `device`

# --- RUNNING STATISTICS STATE ---
mu = None           # Tensor[D], on `device`
M2 = None           # Tensor[D, D], on `device`
n  = 0              # total samples seen

@torch.no_grad()  # disable gradients inside hook
def update_online_cov(x: torch.Tensor):
    """
    x: Tensor of shape (N_batch, D) already on `device` and float32.
    Updates global mu, M2, and n in-place.
    """
    global mu, M2, n

    N, D = x.shape
    n_new = n + N

    if mu is None:
        # First batch
        mu = x.mean(dim=0)                               # [D]
        M2 = (x - mu).t().mm(x - mu)                     # [D, D]
        n = N
        return

    # batch mean
    mb = x.mean(dim=0)                                   # [D]
    delta = mb - mu                                      # [D]
    mu = mu + delta * (N / n_new)                        # [D]

    # within-batch scatter
    dev = x - mb                                         # [N, D]
    M2_batch = dev.t().mm(dev)                           # [D, D]

    # cross term
    cross = torch.outer(delta, delta) * (n * N / n_new)  # [D, D]

    # in-place update
    M2 += M2_batch
    M2 += cross

    n = n_new

def resid_post_hook(resid_post, hook):
    # resid_post: [B, T, D], float32 on CPU by default?
    B, T, D = resid_post.shape
    # encode directly on `device`
    flat = resid_post.view(B * T, D).float().to(device)
    hidden = kronsae.encode(flat)                       # already on device
    update_online_cov(hidden)                            # fast, no transfers
    return resid_post

# --- MAIN LOOP ---
model.to(device)
topk_sae.to(device)
model.eval()                                            # switch to eval mode

# replace streaming=True if you need it
#dataset = load_dataset("HuggingFaceFW/fineweb-edu", split="train", name="CC-MAIN-2024-51", streaming=True, trust_remote_code=True)
iter_ds = iter(dataset)

num_iters = 5000
with torch.no_grad():  # full-loop no-grad
    for _ in tqdm.tqdm(range(num_iters)):
        sample = next(iter_ds)
        tokens = model.to_tokens(sample["text"]).to(device)
        _ = model.run_with_hooks(
            tokens,
            fwd_hooks=[(hook_point, resid_post_hook)]
        )

# --- FINALIZE ---
cov = M2 / (n - 1)                                      # unbiased covariance
std = torch.sqrt(torch.diag(cov))                      # [D]
corrB = cov / (std[:, None] * std[None, :] + 1e-6)      # [D, D]


In [ ]:
d = get_sae("kronsae", "saes/1_5b_qwen/32k_kronsae")
kronsae_random = d["sae"]
torch.nn.init.uniform_(
                kronsae_random.W_dec
            )
torch.nn.init.uniform_(
                kronsae_random.W_enc
            )
torch.nn.init.uniform_(
                kronsae_random.b_dec
            )
torch.nn.init.uniform_(
                kronsae_random.b_enc
            )

In [ ]:
import torch
from datasets import load_dataset

# --- CONFIG ----
device = torch.device(device) #"cuda" if torch.cuda.is_available() else "cpu")
hook_point = utils.get_act_name("resid_post", 14)
# topk_sae = ...        # your SAE encoder, already on `device`
# model = ...           # your model, already on `device`

# --- RUNNING STATISTICS STATE ---
mu = None           # Tensor[D], on `device`
M2 = None           # Tensor[D, D], on `device`
n  = 0              # total samples seen

@torch.no_grad()  # disable gradients inside hook
def update_online_cov(x: torch.Tensor):
    """
    x: Tensor of shape (N_batch, D) already on `device` and float32.
    Updates global mu, M2, and n in-place.
    """
    global mu, M2, n

    N, D = x.shape
    n_new = n + N

    if mu is None:
        # First batch
        mu = x.mean(dim=0)                               # [D]
        M2 = (x - mu).t().mm(x - mu)                     # [D, D]
        n = N
        return

    # batch mean
    mb = x.mean(dim=0)                                   # [D]
    delta = mb - mu                                      # [D]
    mu = mu + delta * (N / n_new)                        # [D]

    # within-batch scatter
    dev = x - mb                                         # [N, D]
    M2_batch = dev.t().mm(dev)                           # [D, D]

    # cross term
    cross = torch.outer(delta, delta) * (n * N / n_new)  # [D, D]

    # in-place update
    M2 += M2_batch
    M2 += cross

    n = n_new

def resid_post_hook(resid_post, hook):
    # resid_post: [B, T, D], float32 on CPU by default?
    B, T, D = resid_post.shape
    # encode directly on `device`
    flat = resid_post.view(B * T, D).float().to(device)
    hidden = kronsae_random.encode(flat)                       # already on device
    update_online_cov(hidden)                            # fast, no transfers
    return resid_post

# --- MAIN LOOP ---
# model.to(device)
# topk_sae.to(device)
model.eval()                                            # switch to eval mode

# replace streaming=True if you need it
#dataset = load_dataset("HuggingFaceFW/fineweb-edu", split="train", name="CC-MAIN-2024-51", streaming=True, trust_remote_code=True)
iter_ds = iter(dataset)

num_iters = 5000
with torch.no_grad():  # full-loop no-grad
    for _ in tqdm.tqdm(range(num_iters)):
        sample = next(iter_ds)
        tokens = model.to_tokens(sample["text"]).to(device)
        _ = model.run_with_hooks(
            tokens,
            fwd_hooks=[(hook_point, resid_post_hook)]
        )

# --- FINALIZE ---
cov = M2 / (n - 1)                                      # unbiased covariance
std = torch.sqrt(torch.diag(cov))                      # [D]
corrB_rand = cov / (std[:, None] * std[None, :] + 1e-6)      # [D, D]


In [ ]:
#torch.save(corr.to(torch.bfloat16), "mul_corr.pt")

In [ ]:
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

A = corrA.float().cpu().numpy() #torch.load('mul_corr.pt', map_location='cpu').float().numpy()
B = corrB.float().cpu().numpy() #torch.load('topk_corr.pt', map_location='cpu').float().numpy()
A_rand = corrA_rand.float().cpu().numpy() #torch.load('mul_corr.pt', map_location='cpu').float().numpy()
B_rand = corrB_rand.float().cpu().numpy() #torch.load('topk_corr.pt', map_location='cpu').float().numpy()

np.fill_diagonal(A, 0)
np.fill_diagonal(B, 0)
np.fill_diagonal(A_rand, 0)
np.fill_diagonal(B_rand, 0)

hist_topk = np.sort(A, axis=1).mean(axis=0)
hist_mul = np.sort(B, axis=1).mean(axis=0)
hist_topk_rand = np.sort(A_rand, axis=1).mean(axis=0)
hist_mul_rand = np.sort(B_rand, axis=1).mean(axis=0)

In [ ]:
from scipy.stats import ks_2samp, kruskal

ks_val, ks_pvalue = ks_2samp(hist_mul, hist_topk)
kw_val, kw_pvalue = kruskal(hist_mul, hist_topk)

ks_val, ks_pvalue, kw_val, kw_pvalue

In [ ]:
ks_val, ks_pvalue = ks_2samp(hist_mul_rand, hist_topk_rand)
kw_val, kw_pvalue = kruskal(hist_mul_rand, hist_topk_rand)

ks_val, ks_pvalue, kw_val, kw_pvalue

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming hist_mul and hist_topk are your data arrays
fig, ax = plt.subplots(1, 1, figsize=(8, 5), dpi=250)

palette = sns.color_palette('tab10')

#Plot both positive and negative values together using symlog scale
sns.histplot(hist_mul,   log_scale=False, stat="density", bins=256 + 128, 
             label="KronSAE", ax=ax, element="step", fill=False, color=palette[0])
sns.histplot(hist_topk, log_scale=False, stat="density", bins=256 + 128, 
             label="TopK", ax=ax, element="step", fill=False, color=palette[1])


# Set symmetric logarithmic x-axis
ax.set_xscale('symlog', linthresh=2e-3)  # Adjust linthresh based on your data

# scales = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11]
# scales = [-s for s in scales] + [0] + list(reversed(scales))

# Customize ticks to show negative/positive explicitly
# ax.set_xticks(scales)
# ax.set_xticklabels([f'{s:.0e}' for s in scales])

# Log scale for y-axis to see density differences
ax.set_yscale('log')
ax.grid(alpha=0.35)
# ax.set_title('Positive & Negative Values (Symmetric Log Scale)')
ax.legend(loc='upper right')

# plt.axvline(np.median(hist_mul), linestyle='--',  color=palette[0])
# plt.axvline(np.median(hist_topk), linestyle='--',  color=palette[1])

plt.tight_layout()
plt.savefig(f'sae_acts_corr_distrib_qwen1.5b_final.pdf')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming hist_mul and hist_topk are your data arrays
fig, ax = plt.subplots(1, 1, figsize=(8, 6)) #, dpi=250)

palette = sns.color_palette('tab10') #[2:]

def proc_func(x):
    x_abs = np.sort(np.abs(x)) #[::-1]
    #x_abs = np.cumsum(x_abs) 
    x_abs /= x_abs.max()
    return x_abs

sns.histplot(proc_func(hist_mul_rand),   log_scale=False, stat="density", bins=256 + 128, 
             label="KronSAE (random)", ax=ax, 
             element="step", 
             fill=True, 
             color=palette[2])
sns.histplot(proc_func(hist_topk_rand), log_scale=False, stat="density", bins=256 + 128, 
             label="TopK (random)", ax=ax, 
             element="step", 
             fill=True, 
             color=palette[3])

sns.histplot(proc_func(hist_mul),   log_scale=False, stat="density", bins=256 + 128, 
             label="KronSAE", ax=ax, 
             element="step", 
             fill=True, 
             color=palette[0])
sns.histplot(proc_func(hist_topk), log_scale=False, stat="density", bins=256 + 128, 
             label="TopK", ax=ax, 
             element="step", 
             fill=True,
             color=palette[1])
#Plot both positive and negative values together using symlog scale



# Set symmetric logarithmic x-axis
ax.set_xscale('symlog', linthresh=4e-1)  # Adjust linthresh based on your data

# scales = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11]
# scales = [-s for s in scales] + [0] + list(reversed(scales))

# Customize ticks to show negative/positive explicitly
# ax.set_xticks(scales)
# ax.set_xticklabels([f'{s:.0e}' for s in scales])

# Log scale for y-axis to see density differences
ax.set_yscale('log')
ax.grid(alpha=0.35)
# ax.set_title('Positive & Negative Values (Symmetric Log Scale)')
ax.legend(loc='upper right')

plt.axvline(np.mean(proc_func(hist_mul)), linestyle='--',  color=palette[0])
plt.axvline(np.mean(proc_func(hist_topk)), linestyle='--',  color=palette[1])
plt.axvline(np.mean(proc_func(hist_mul_rand)), linestyle='--',  color=palette[2])
plt.axvline(np.mean(proc_func(hist_topk_rand)), linestyle='--',  color=palette[3])

plt.tight_layout()
#plt.savefig(f'rand_sae_acts_corr_distrib_qwen1.5b.pdf')
plt.show()

In [ ]:
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Example usage:
def get_corr_distrib(corr_marix):
    corr_matrix = torch.tensor(corr_marix)
    group_size = 16
    max_samples: int = 100000
    random_seed: int = 42
    """
    Compare within-group and between-group correlation distributions.
    
    Parameters:
    - corr_matrix: torch.Tensor of shape (N, N), correlation matrix.
    - group_size: int, size k of each group (assumed contiguous blocks).
    - max_samples: int, maximum number of samples per distribution to plot/compare.
    - random_seed: int, for reproducibility of sampling.
    
    Returns:
    - t_stat: float, t-test statistic comparing inner vs. between.
    - p_value: float, two-sided p-value of the test.
    """
    torch.manual_seed(random_seed)
    N = corr_matrix.size(0)
    if N % group_size != 0:
        raise ValueError("Matrix size must be divisible by group_size")
    num_groups = N // group_size
    
    inner_vals = []
    between_vals = []
    
    # Collect samples
    for g in range(num_groups):
        s1 = g * group_size
        e1 = s1 + group_size
        block = corr_matrix[s1:e1, s1:e1]
        triu = torch.triu_indices(group_size, group_size, offset=1)
        inner_vals.append(block[triu[0], triu[1]])
    
        # sample sqrt(num_groups) other groups
        others = list(range(num_groups))
        others.remove(g)
        k_sample = min(len(others), int(np.sqrt(num_groups)))
        chosen = np.random.choice(others, size=k_sample, replace=False)
        for h in chosen:
            s2 = h * group_size
            e2 = s2 + group_size
            between_vals.append(corr_matrix[s1:e1, s2:e2].flatten())
    
    inner = torch.cat(inner_vals)
    between = torch.cat(between_vals)
    
    # subsample if needed
    def subsample(tensor):
        total = tensor.numel()
        if total > max_samples:
            idx = torch.randperm(total)[:max_samples]
            return tensor[idx]
        return tensor
    
    inner = subsample(inner)
    between = subsample(between)
    
    inner_np = inner.cpu().numpy()
    between_np = between.cpu().numpy()
    
    combined = np.concatenate((inner_np, between_np))
    min_val, max_val = combined.min(), combined.max()
    num_bins = 36
    bins = np.linspace(min_val, max_val, num_bins + 1)
    return inner_np, between_np, bins


In [ ]:
inner_np, between_np, bins = get_corr_distrib(B)

In [ ]:
inner_np_rand, between_np_rand, bins_rand = get_corr_distrib(B_rand)

In [ ]:
# Plot both histograms on one axes
plt.figure(figsize=(8, 6), dpi=250)

palette = sns.color_palette('tab10')[2:]

#Plot both positive and negative values together using symlog scale

sns.histplot(inner_np, bins=bins,  kde=False, stat="density", label='Within-Group', alpha=0.6, color=palette[0])
sns.histplot(between_np, bins=bins,  kde=False, stat="density", label='Between-Group', alpha=0.6, color=palette[1])

# Avoid zero-density dominating by setting bins away from zero and using logarithmic scales
#plt.xscale('symlog')
plt.yscale('log')
plt.xlabel('Correlation Value')
plt.ylabel('Density (log scale)')
plt.title('Within vs. Between Group Correlation Distributions')
plt.legend()
plt.tight_layout()
plt.savefig(f'sae_acts_head_corr_distrib_qwen1.5b_final.pdf')
plt.show()

# t-test
t_stat, p_value = ttest_ind(inner_np, between_np, equal_var=False)
print(f"T-statistic: {t_stat:.4f}, p-value: {p_value:.4e}")


In [ ]:
# Plot both histograms on one axes
plt.figure(figsize=(8, 5), dpi=250)

palette = sns.color_palette('tab10')

#Plot both positive and negative values together using symlog scale

sns.histplot(inner_np, bins=bins,  kde=False, stat="density", label='Within-Group', alpha=0.3, color=palette[0], element="step", linewidth=1.5  )
sns.histplot(between_np, bins=bins,  kde=False, stat="density", label='Between-Group', alpha=0.3, color=palette[1], element="step", linewidth=1.5 )
sns.histplot(inner_np_rand, bins=bins,  kde=False, stat="density", label='Within-Group (rand)', alpha=0.99, color=palette[0], element="step", fill=False, linestyle='--', linewidth=1.5 )
sns.histplot(between_np_rand, bins=bins,  kde=False, stat="density", label='Between-Group (rand)', alpha=0.99, color=palette[1], element="step", fill=False, linestyle='--', linewidth=1.5 )
m1 = np.mean(inner_np)
m2 = np.mean(between_np)
m3 = np.mean(inner_np_rand)
m4 = np.mean(between_np_rand)

# plt.axvline(m1, linestyle='-',  alpha=0.6,color=palette[0])
# plt.axvline(m2, linestyle='-',  alpha=0.6,color=palette[1])
# plt.axvline(m3, linestyle='--',  alpha=0.99, color=palette[0])
# plt.axvline(m4, linestyle='--',   alpha=0.99, color=palette[1])

# Avoid zero-density dominating by setting bins away from zero and using logarithmic scales
#plt.xscale('symlog')
plt.yscale('log')
plt.xlabel('Pearson correlation', fontsize=14)
plt.ylabel('Density (log scale)', fontsize=14)
#plt.title('Within vs. Between Group Correlation Distributions')
plt.legend()
plt.tight_layout()
plt.savefig(f'rand_sae_acts_head_corr_distrib_qwen1.5b.pdf')
plt.show()
print(m1, 
m2 ,
m3 ,
m4 ,)
# t-test
t_stat, p_value = ttest_ind(inner_np, between_np, equal_var=False)
print(f"T-statistic: {t_stat:.4f}, p-value: {p_value:.4e}")

In [ ]:
m1 = np.median(inner_np)
m2 = np.median(between_np)
m3 = np.median(inner_np_rand)
m4 = np.median(between_np_rand)
m1,m2,m3,m4